# 连续批处理：CPU 调度模拟

此 notebook **不加载模型、不使用 GPU**。它只模拟请求何时到达、何时完成，以及静态批处理与连续批处理留下多少空槽。

In [ ]:
from dataclasses import dataclass

@dataclass
class Request:
    name: str
    arrival: int
    remaining_tokens: int

WORKLOAD = [Request('A', 0, 2), Request('B', 0, 7), Request('C', 1, 3), Request('D', 3, 4)]
CAPACITY = 2

In [ ]:
def simulate(continuous: bool, capacity: int = CAPACITY):
    waiting = [Request(r.name, r.arrival, r.remaining_tokens) for r in WORKLOAD]
    running, completed, tick, empty_slots = [], {}, 0, 0
    print('模式：' + ('连续批处理' if continuous else '静态批处理'))
    while waiting or running:
        arrived = [r for r in waiting if r.arrival <= tick]
        if continuous:
            waiting = [r for r in waiting if r.arrival > tick]
            while arrived and len(running) < capacity:
                running.append(arrived.pop(0))
            waiting = arrived + waiting
        elif not running and arrived:
            waiting = [r for r in waiting if r.arrival > tick]
            running = arrived[:capacity]
            waiting = arrived[capacity:] + waiting
        names = ', '.join(r.name for r in running) or '（空）'
        print(f't={tick:>2}: 运行 [{names}]，空槽 {capacity - len(running)}')
        empty_slots += capacity - len(running)
        for r in running:
            r.remaining_tokens -= 1
        done = [r for r in running if r.remaining_tokens == 0]
        for r in done:
            completed[r.name] = tick + 1
            running.remove(r)
        tick += 1
    print('完成时间：', completed, '；累计空槽：', empty_slots)
    return completed, empty_slots

In [ ]:
static = simulate(continuous=False)
print()
continuous = simulate(continuous=True)

assert continuous[1] <= static[1], '在此工作负载中连续批处理应不留下更多空槽'
print('\n尝试修改 WORKLOAD、CAPACITY 或到达时间；这不是 GPU 性能测试。')